In [ ]:
from fractions import Fraction
from pathlib import Path
from typing import List, Union
from harmonic_inference.data.corpus_constants import MEASURE_OFFSET
import music21
import pandas as pd
import numpy as np
from music21.converter import parse
from music21.harmony import ChordSymbol
from music21.stream import Measure, Stream


In [ ]:
def get_chords_and_measures_df_from_m21_score(m21_score: music21.stream.Score) -> pd.DataFrame:
    """
    a script which takes an annotated .xml/.mxl file as input and maps it into a .tsv file.
    """
    """
    Parameters
    ----------
    m21_score : music21.stream.Score
        A music21 Score that has been parsed already.

    Returns
    -------
    1. measures_df : pd.DataFrame
        A measures_df with the following columns:
            'mc' (int): The measure index.
            "annotations": The harmonic annotation of each measure.
            'timesig' (str): The time signature of each measure.
            'start' (Fraction): The "offset" position at the start of each measure, in
                                whole notes since the beginning of the piece.
            'act_dur' (Fraction): The duration of the measure, in whole notes.
            'mc_offset' (Fraction): The starting position of this measure, in whole notes
                                 after the most recent downbeat.
            'next' (int): The measure index of the measure that follows each one.

    2. chords_df : pd.DataFrame
        A chords_df with the following columns:
            'mn' (int): The measure number
            'annotations' (str): The annotations of each measure
    """
    # Lists to compute and add to output
    time_signatures = []
    # annotations = []
    starts = []
    lengths = []
    df_offsets = []
    mns = []

    # The start of the 2nd measure (the first full measure in the case of an anacrusis)
    ts_epoch = Fraction(list(m21_score.measureOffsetMap().keys())[1] / 4)

    # Default time signature
    time_signature = "4/4"
    ts_duration = Fraction(time_signature)

    # First and last measure number (inclusive) of all first endings
    first_endings = [
        (bracket.getFirst().number, bracket.getLast().number)
        for bracket in m21_score.flat.getElementsByClass(music21.spanner.RepeatBracket)
        if bracket.number.startswith("1")
    ]
    skipped_dur = 0
    
    # J: Add Annotations to a list

    chord_symbols = []
    chords_by_measure = []
    for element in m21_score.recurse().getElementsByClass(ChordSymbol):
        measure = element.getContextByClass("Measure")
        # element.activeSite.remove(element) # Do we need that?
        chord_symbols.append(element.figure)# OR str(element) OR element.pitchedCommonName
        chords_by_measure.append(element.measureNumber) 
        # print(element, measure)
    # print(chord_symbols, chords_by_measure) # for checking

    # Go through the measures and add them to the tracking lists
    for mc, (offset, measures_list) in enumerate(m21_score.measureOffsetMap().items()):
        offset = Fraction(offset) / 4 - skipped_dur
        measure = measures_list[0]

        # chord_symbols = []
        # for element in measure.recurse().getElementsByClass(ChordSymbol):
        #     # element.activeSite.remove(element)
        #     chord_symbols.append(element)

        skip = False
        for start, end in first_endings:
            if measure.measureNumber in range(start, end + 1):
                skip = True
                break
        if skip:
            skipped_dur += Fraction(measure.duration.quarterLength) / 4
            continue

        if measure.timeSignature is not None:
            
            if measure.timeSignature.ratioString != time_signature:
                # Time Signature change
                time_signature = measure.timeSignature.ratioString
                ts_duration = Fraction(time_signature)

                # Reset the ts_epoch to this location
                if mc != 0:
                    ts_epoch = offset



        if lengths:
            # Set the length of each bar to the difference between consecutive measure offsets
            lengths[-1] = offset - starts[-1]
        # Default (used only for the last measure)
        lengths.append(Fraction(measure.duration.quarterLength) / 4)
        # annotations.append(chord_symbols)
        starts.append(offset)
        time_signatures.append(time_signature)
        df_offsets.append((offset - ts_epoch) % ts_duration)
        mns.append(measure.measureNumber)
        
    mcs = list(range(len(mns)))

    return pd.DataFrame(
        {
            "mc": mcs,
            "mn": mns,
            "timesig": time_signatures,
            "start": starts,
            "act_dur": lengths,
            MEASURE_OFFSET: df_offsets,
            "next": mcs[1:] + [pd.NA],
        }
    ), pd.DataFrame(
        {
        # "mc": mcs,
        "mn": chords_by_measure,
        "annotations": chord_symbols
        }
    )


"""
tsv conversion function
"""
def score_to_tsv(
    music_xml_path: Union[Path, str],
    output_dir: Union[Path, str] = None
):
    m21_score: Stream = parse(music_xml_path)
    measures_df, chords_df = get_chords_and_measures_df_from_m21_score(m21_score)
    
    if output_dir is None:
        output_dir = Path(music_xml_path).parent
    else:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
    
    # Auto-generate filename from input
    output_path = output_dir / f"{Path(music_xml_path).stem}.tsv"
    
    chords_df.to_csv(output_path, sep="\t", index=False)
    return measures_df, chords_df

In [50]:
score_to_tsv(
    music_xml_path="mels_to_harmonize/score_192-Qualquer_coisa-Irineu_de_Almeida.xml",
    output_dir="tests_tsv"  # folder only
)

(    mc  mn timesig start act_dur mc_offset  next
 0    0   1     2/4     0     1/2         0     1
 1    1   2     2/4   1/2     1/2         0     2
 2    2   3     2/4     1     1/2         0     3
 3    3   4     2/4   3/2     1/2         0     4
 4    4   5     2/4     2     1/2         0     5
 5    5   6     2/4   5/2     1/2         0     6
 6    6   7     2/4     3     1/2         0     7
 7    7   8     2/4   7/2     1/2         0     8
 8    8   9     2/4     4     1/2         0     9
 9    9  10     2/4   9/2     1/2         0    10
 10  10  11     2/4     5     1/2         0    11
 11  11  12     2/4  11/2     1/2         0    12
 12  12  13     2/4     6     1/2         0    13
 13  13  14     2/4  13/2     1/2         0    14
 14  14  15     2/4     7     1/2         0    15
 15  15  17     2/4  15/2     1/2         0    16
 16  16  18     2/4     8     1/2         0    17
 17  17  19     2/4  17/2     1/2         0    18
 18  18  20     2/4     9     1/2         0    19


In [16]:
# manual tests

m21_score = parse("mels_to_harmonize/score_192-Qualquer_coisa-Irineu_de_Almeida.xml")
existing_chord_symbols = []

for element in m21_score.recurse().getElementsByClass(ChordSymbol):
    # Get the measure object this chord belongs to
    measure = element.getContextByClass('Measure')
    
    if measure is not None:
        print(f"Chord: {element}, Measure Number: {measure.number}")
    else:
        print(f"Chord: {element} is not in a measure.")

    # remove and store
    element.activeSite.remove(element)
    existing_chord_symbols.append(element)
existing_chord_symbols

Chord: <music21.harmony.ChordSymbol F>, Measure Number: 1
Chord: <music21.harmony.ChordSymbol D7>, Measure Number: 1
Chord: <music21.harmony.ChordSymbol G7>, Measure Number: 2
Chord: <music21.harmony.ChordSymbol C7>, Measure Number: 3
Chord: <music21.harmony.ChordSymbol F>, Measure Number: 4
Chord: <music21.harmony.ChordSymbol F>, Measure Number: 5
Chord: <music21.harmony.ChordSymbol Am>, Measure Number: 6
Chord: <music21.harmony.ChordSymbol E7>, Measure Number: 7
Chord: <music21.harmony.ChordSymbol C7>, Measure Number: 8
Chord: <music21.harmony.ChordSymbol F>, Measure Number: 9
Chord: <music21.harmony.ChordSymbol D7>, Measure Number: 9
Chord: <music21.harmony.ChordSymbol G7>, Measure Number: 10
Chord: <music21.harmony.ChordSymbol C7>, Measure Number: 11
Chord: <music21.harmony.ChordSymbol F>, Measure Number: 12
Chord: <music21.harmony.ChordSymbol D7>, Measure Number: 12
Chord: <music21.harmony.ChordSymbol Gm>, Measure Number: 13
Chord: <music21.harmony.ChordSymbol G#dim>, Measure Numb

[<music21.harmony.ChordSymbol F>,
 <music21.harmony.ChordSymbol D7>,
 <music21.harmony.ChordSymbol G7>,
 <music21.harmony.ChordSymbol C7>,
 <music21.harmony.ChordSymbol F>,
 <music21.harmony.ChordSymbol F>,
 <music21.harmony.ChordSymbol Am>,
 <music21.harmony.ChordSymbol E7>,
 <music21.harmony.ChordSymbol C7>,
 <music21.harmony.ChordSymbol F>,
 <music21.harmony.ChordSymbol D7>,
 <music21.harmony.ChordSymbol G7>,
 <music21.harmony.ChordSymbol C7>,
 <music21.harmony.ChordSymbol F>,
 <music21.harmony.ChordSymbol D7>,
 <music21.harmony.ChordSymbol Gm>,
 <music21.harmony.ChordSymbol G#dim>,
 <music21.harmony.ChordSymbol F>,
 <music21.harmony.ChordSymbol C7>,
 <music21.harmony.ChordSymbol F>,
 <music21.harmony.ChordSymbol F>,
 <music21.harmony.ChordSymbol C>,
 <music21.harmony.ChordSymbol D7>,
 <music21.harmony.ChordSymbol G7>,
 <music21.harmony.ChordSymbol C>,
 <music21.harmony.ChordSymbol C>,
 <music21.harmony.ChordSymbol G>,
 <music21.harmony.ChordSymbol B7>,
 <music21.harmony.ChordSymbol